# WaZP Stacked ΔΣ Signals  (3D nested binning)

Produce **27 disjoint** stacked ΔΣ profiles for the WaZP real-data catalog by **nesting**:

1. **3 redshift slices** at fixed, literature-style edges in `ZPHOT`.
2. Within each z-slice → **3 richness slices** at fixed edges in `NGALS` (same edges across z).
3. Within each (z, NGALS) cell → **3 Δm tertiles** computed **per-cell** (equal-population terciles of finite `DELTA_M_I` *inside that cell*, so cell-local 33/33/33).

Each cluster belongs to **at most one** of the 27 stacks (clusters with non-finite `DELTA_M_I`, or outside the z/NGALS edges, are excluded — they have nowhere to live).

Pipeline:

1. **Load** the WaZP real-data catalog (§1).
2. **Load** the metacal source pre-filter from `wazp_lensing_signals.ipynb` (§2).
3. **BallTree** over source positions for cone queries (§3).
4. **Nested bin assignment** (§4): assign each cluster a `(z_bin, ng_bin, dm_bin)` triple.
5. **`bkg_worker`** per cluster (§5).
6. **Parallel `bkg` computation** with `multiprocess.Pool` (§6).
7. **Stack each of the 27 cells** with `xlensing.data.stacked_signal` (§7).
8. **Save & plot** as a 3×3 panel grid with 3 Δm overlays per panel (§8).

This notebook shares `data/metacal_filtered.npz` with `wazp_lensing_signals.ipynb`.

In [ ]:
import os, time, pickle, warnings
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table
from astropy.cosmology import Planck18 as COSMO
from astropy import units as u
from sklearn.neighbors import BallTree
from multiprocess import Pool, cpu_count
import xlensing.data
warnings.filterwarnings('ignore')

WAZP_REAL    = 'data/wazp_real_data.fits'
FILTERED_NPZ = 'data/metacal_filtered.npz'
RESULTS_PKL  = 'data/wazp_stacked_signals.pkl'
BKG_CACHE    = 'data/wazp_bkgs.pkl'   # per-cluster bkg cache

# Pipeline parameters --------------------------------------------------------
USE_CACHE  = True          # set to False to force a clean rebuild of every intermediate
RADIUS_MPC = 5.0
NBOOT      = 500
NWORKERS   = max(1, cpu_count() // 2)

# Radial bins (Mpc) — same as wazp_lensing_signals.ipynb / prototype ---------
radii  = np.logspace(-0.6, 0.6, 8)
_N     = len(radii)
_edges = np.logspace(
    np.log10(radii[0])   + (np.log10(radii[0]) - np.log10(radii[1])) / 2,
    np.log10(radii[-1])  - (np.log10(radii[0]) - np.log10(radii[1])) / 2,
    _N + 1,
)
BINS_LIMS = np.array([[_edges[i], _edges[i+1]] for i in range(_N)])

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
os.makedirs('data', exist_ok=True)

print(f'USE_CACHE = {USE_CACHE}')
print(f'NWORKERS  = {NWORKERS}')
print(f'NBOOT     = {NBOOT}')

## 1 — Load the WaZP real-data catalog

In [2]:
wazp = Table.read(WAZP_REAL)
print(f'Loaded {len(wazp):,} clusters from {WAZP_REAL}')
print(f'Columns: {wazp.colnames}')
print(f'  ZPHOT range: {wazp["ZPHOT"].min():.3f} – {wazp["ZPHOT"].max():.3f}')
print(f'  NGALS range: {wazp["NGALS"].min():.1f} – {wazp["NGALS"].max():.1f}')
n_finite = int(np.isfinite(wazp["DELTA_M_I"]).sum())
print(f'  finite DELTA_M_I: {n_finite:,} / {len(wazp):,}')

Loaded 15,967 clusters from data/wazp_real_data.fits
Columns: ['ID', 'RA', 'DEC', 'NGALS', 'ZPHOT', 'DELTA_M_I', 'DELTA_M_I_ERR']
  ZPHOT range: 0.057 – 0.700
  NGALS range: 25.0 – 294.7
  finite DELTA_M_I: 15,967 / 15,967


## 2 — Load the metacal source pre-filter

Reuses the intermediate produced by `wazp_lensing_signals.ipynb`. If the file does not yet exist, run that notebook's §2 cell first (or wait for it to finish).

In [3]:
if not os.path.exists(FILTERED_NPZ):
    raise FileNotFoundError(
        f'{FILTERED_NPZ} not found. Run wazp_lensing_signals.ipynb §2 first '
        f'(or wait for it to finish if it is currently running).'
    )

f = np.load(FILTERED_NPZ)
SR_RA  = f['ra'];  SR_DEC = f['dec']; SR_Z = f['z']
SR_E1  = f['e1'];  SR_E2  = f['e2'];  SR_W = f['w']
SR_R11 = f['R11']; SR_R12 = f['R12']
SR_R21 = f['R21']; SR_R22 = f['R22']
print(f'Loaded {len(SR_RA):,} pre-filtered sources')

Loaded 126,719,902 pre-filtered sources


## 3 — BallTree spatial index

In [4]:
t0 = time.time()
TREE = BallTree(np.column_stack([SR_DEC, SR_RA]), metric='haversine')
print(f'BallTree built in {time.time()-t0:.1f}s')

BallTree built in 67.9s


## 4 — Nested bin assignment

- **Redshift** (fixed): `[0.10, 0.35, 0.55, 0.70]` — three slices covering the catalog's `ZPHOT < 0.7` cut.
- **Richness** (fixed, same edges across z): `[25, 40, 70, ∞]` — low/mid/high.
- **Δm tertile** computed **per (z, richness) cell**: equal-population 33/33/33 split of finite `DELTA_M_I` *inside that cell*. So Δm-bin "0" in cell (0,0) is a different `DELTA_M_I` interval than Δm-bin "0" in cell (2,2).

This builds a disjoint 3×3×3 partition. Clusters with non-finite `DELTA_M_I` or outside the z/richness edges are flagged with bin = −1 and excluded from all 27 stacks.

In [5]:
z_edges  = np.array([0.10, 0.35, 0.55, 0.70])
ng_edges = np.array([25.0, 40.0, 70.0, np.inf])

def bin_idx(values, edges):
    """Assign each value to a bin (0..len(edges)-2), or -1 if outside [edges[0], edges[-1]]."""
    v   = np.asarray(values, dtype=float)
    idx = np.digitize(v, edges[1:-1], right=False)
    out = ~((v >= edges[0]) & (v <= edges[-1]))
    idx[out] = -1
    return idx

z_bin  = bin_idx(wazp['ZPHOT'], z_edges)
ng_bin = bin_idx(wazp['NGALS'], ng_edges)
fin_dm = np.isfinite(wazp['DELTA_M_I'])

# Per-cell Δm tertile edges and per-cluster Δm-bin
dm_bin           = np.full(len(wazp), -1, dtype=int)
dm_edges_per_cell = {}                  # (i, j) -> array shape (4,)

for i in range(3):
    for j in range(3):
        cell = (z_bin == i) & (ng_bin == j) & fin_dm
        n    = int(cell.sum())
        if n < 3:                       # too few to split into terciles
            dm_edges_per_cell[(i, j)] = None
            continue
        cell_dm = np.asarray(wazp['DELTA_M_I'])[cell]
        edges   = np.quantile(cell_dm, [0.0, 1/3, 2/3, 1.0])
        dm_edges_per_cell[(i, j)] = edges
        # local 0/1/2 assignment within this cell, then scatter to global
        local       = np.digitize(cell_dm, edges[1:-1], right=False)
        dm_bin[np.where(cell)[0]] = local

# Diagnostic table
print(f'{"cell":>6}  {"z":>14}  {"NGALS":>14}  '
      f'{"N(z,ng)":>9}  {"N stacked":>9}  '
      f'{"Δm0":>16}  {"Δm1":>16}  {"Δm2":>16}')
print('-' * 130)
total = 0
for i in range(3):
    for j in range(3):
        cell_zr = (z_bin == i) & (ng_bin == j)
        cell_all= cell_zr & fin_dm
        n_cell  = int(cell_zr.sum())
        n_use   = int(cell_all.sum())
        total  += n_use
        edges   = dm_edges_per_cell[(i, j)]
        if edges is None:
            d0 = d1 = d2 = '—'
        else:
            d0 = f'[{edges[0]:+.2f},{edges[1]:+.2f})'
            d1 = f'[{edges[1]:+.2f},{edges[2]:+.2f})'
            d2 = f'[{edges[2]:+.2f},{edges[3]:+.2f}]'
        zr  = f'[{z_edges[i]:.2f},{z_edges[i+1]:.2f})'
        ngr = (f'[{ng_edges[j]:.0f},{ng_edges[j+1]:.0f})'
               if np.isfinite(ng_edges[j+1])
               else f'[{ng_edges[j]:.0f},inf)')
        print(f'{"("+str(i)+","+str(j)+")":>6}  {zr:>14}  {ngr:>14}  '
              f'{n_cell:>9,}  {n_use:>9,}  '
              f'{d0:>16}  {d1:>16}  {d2:>16}')
print('-' * 130)
print(f'Total clusters going into the 27 stacks: {total:,}  '
      f'(excluded: {len(wazp)-total:,})')

  cell               z           NGALS    N(z,ng)  N stacked               Δm0               Δm1               Δm2
----------------------------------------------------------------------------------------------------------------------------------
 (0,0)     [0.10,0.35)         [25,40)      1,567      1,567     [-0.50,+0.45)     [+0.45,+0.93)     [+0.93,+2.64]
 (0,1)     [0.10,0.35)         [40,70)        751        751     [-0.33,+0.50)     [+0.50,+1.04)     [+1.04,+2.89]
 (0,2)     [0.10,0.35)        [70,inf)        295        295     [-0.18,+0.49)     [+0.49,+0.97)     [+0.97,+2.05]
 (1,0)     [0.35,0.55)         [25,40)      4,940      4,940     [-0.46,+0.42)     [+0.42,+0.89)     [+0.89,+2.64]
 (1,1)     [0.35,0.55)         [40,70)      2,170      2,170     [-0.18,+0.47)     [+0.47,+0.94)     [+0.94,+2.54]
 (1,2)     [0.35,0.55)        [70,inf)        698        698     [-0.10,+0.45)     [+0.45,+0.95)     [+0.95,+2.39]
 (2,0)     [0.55,0.70)         [25,40)      3,748      3,748    

## 5 — Per-cluster `bkg_worker`

Same cone-query + `metacal_cluster_lensing` call as `wazp_lensing_signals.ipynb`, but here we stop at the per-cluster `bkg` array (the 6-row stack `[Σ_crit, e_t, e_x, W, R, M]` of the background sources). Stacking is done in §7 once we have all `bkg`s.

In [6]:
def bkg_worker(row):
    """row: dict with 'ID', 'RA' (deg), 'DEC' (deg), 'ZPHOT'.
    Returns dict with 'ID', 'bkg' (float32 6xN), 'count', or 'error'."""
    try:
        cid   = int(row['ID'])
        z_cl  = float(row['ZPHOT'])
        ra_r  = np.deg2rad(float(row['RA']))
        dec_r = np.deg2rad(float(row['DEC']))

        D_A = float(COSMO.angular_diameter_distance(z_cl).to_value(u.Mpc))
        if D_A <= 0:
            return {'ID': cid, 'error': f'D_A={D_A}'}
        ang_r = RADIUS_MPC / D_A

        idx = TREE.query_radius(np.array([[dec_r, ra_r]]), r=ang_r)[0]
        if len(idx) == 0:
            return {'ID': cid, 'bkg': None, 'count': 0}

        sources = (
            SR_RA [idx], SR_DEC[idx], SR_Z [idx],
            SR_E1 [idx], SR_E2 [idx], SR_W [idx],
            SR_R11[idx], SR_R12[idx], SR_R21[idx], SR_R22[idx],
        )
        res = xlensing.data.metacal_cluster_lensing(
            (ra_r, dec_r, z_cl), sources, radius=RADIUS_MPC,
        )
        if res['Count'] == 0:
            return {'ID': cid, 'bkg': None, 'count': 0}

        bkg = np.array([
            np.asarray(res['Critical Density'], dtype='float32'),
            np.asarray(res['Tangential Shear'], dtype='float32'),
            np.asarray(res['Cross Shear'],      dtype='float32'),
            np.asarray(res['Weights'],          dtype='float32'),
            np.asarray(res['Radial Distance'],  dtype='float32'),
            np.asarray(res['Mult. Bias'],       dtype='float32'),
        ])
        return {'ID': cid, 'bkg': bkg, 'count': int(res['Count'])}
    except Exception as exc:
        return {'ID': int(row.get('ID', -1)), 'error': repr(exc)}

## 6 — Parallel `bkg` computation

Reads-from-cache if a prior run already produced `BKG_CACHE`; otherwise runs the pool. The cache lets us iterate on the stacking step (§7) without redoing the metacal lookup.

In [ ]:
if os.path.exists(BKG_CACHE) and USE_CACHE:
    with open(BKG_CACHE, 'rb') as fh:
        bkg_results = pickle.load(fh)
    print(f'Loaded {len(bkg_results):,} cached bkg results from {BKG_CACHE}')
else:
    if os.path.exists(BKG_CACHE):
        print(f'USE_CACHE = False, rebuilding {BKG_CACHE} from scratch')
    input_list = [
        {'ID': int(wazp['ID'][i]),    'RA':   float(wazp['RA'][i]),
         'DEC':float(wazp['DEC'][i]), 'ZPHOT':float(wazp['ZPHOT'][i])}
        for i in range(len(wazp))
    ]
    print(f'Computing bkgs for {len(input_list):,} clusters with {NWORKERS} workers')

    bkg_results = []
    t0 = time.time()
    with Pool(NWORKERS) as pool:
        for i, r in enumerate(pool.imap_unordered(
                bkg_worker, input_list, chunksize=4)):
            bkg_results.append(r)
            if (i + 1) % 200 == 0 or (i + 1) == len(input_list):
                dt  = time.time() - t0
                eta = dt * (len(input_list) - i - 1) / (i + 1)
                print(f'  {i+1:>6,}/{len(input_list):,}  '
                      f'elapsed {dt/60:5.1f} min  eta {eta/60:5.1f} min',
                      flush=True)
            if (i + 1) % 2000 == 0:
                with open(BKG_CACHE + '.partial', 'wb') as fh:
                    pickle.dump(bkg_results, fh)
    with open(BKG_CACHE, 'wb') as fh:
        pickle.dump(bkg_results, fh)
    print(f'\nDone in {(time.time()-t0)/60:.1f} min, cached to {BKG_CACHE}')

# Build ID -> bkg lookup for stacking
bkg_by_id = {r['ID']: r['bkg'] for r in bkg_results
             if r.get('bkg') is not None}
print(f'Usable bkgs: {len(bkg_by_id):,}')

## 7 — Stack each of the 27 cells with `xlensing.data.stacked_signal`

Iterate the disjoint 3×3×3 partition. Stack name: `z{i}_ng{j}_dm{k}`. For each cell we keep `(sigmas, boosts, sigmas_cov, xigmas, xigmas_cov)` plus bin metadata.

In [ ]:
def collect_bkgs(mask):
    ids = np.asarray(wazp['ID'])[mask]
    return [bkg_by_id[int(i)] for i in ids if int(i) in bkg_by_id]

# stacked_signal now returns boost-corrected ΔΣ + full propagated covariance
# (joint cluster bootstrap of boost and signal).
stacks = {}
boost_outer = []
for i in range(3):
    for j in range(3):
        for k in range(3):
            name = f'z{i}_ng{j}_dm{k}'
            mask = (z_bin == i) & (ng_bin == j) & (dm_bin == k)
            n_in = int(mask.sum())
            if n_in == 0:
                print(f'{name}: empty, skipping')
                continue
            bkgs = collect_bkgs(mask)
            if not bkgs:
                print(f'{name}: no bkgs available, skipping')
                continue

            t0 = time.time()
            sigmas, boosts, boosts_cov, sigmas_cov, xigmas, xigmas_cov = (
                xlensing.data.stacked_signal(bkgs, BINS_LIMS, NBOOT)
            )

            dm_e = dm_edges_per_cell[(i, j)]
            stacks[name] = {
                'z_bin':     i,
                'ng_bin':    j,
                'dm_bin':    k,
                'z_lo':      float(z_edges[i]),
                'z_hi':      float(z_edges[i+1]),
                'ng_lo':     float(ng_edges[j]),
                'ng_hi':     float(ng_edges[j+1]),
                'dm_lo':     float(dm_e[k])   if dm_e is not None else np.nan,
                'dm_hi':     float(dm_e[k+1]) if dm_e is not None else np.nan,
                'n_in_cell': n_in,
                'n_stacked': len(bkgs),
                'sigmas':     np.asarray(sigmas),
                'sigmas_cov': np.asarray(sigmas_cov),
                'boosts':     np.asarray(boosts),
                'boosts_cov': np.asarray(boosts_cov),
                'xigmas':     np.asarray(xigmas),
                'xigmas_cov': np.asarray(xigmas_cov),
            }
            b_err_outer = float(np.sqrt(boosts_cov[-1, -1]))
            boost_outer.append((name, float(boosts[-1]), b_err_outer))
            print(f'{name}: '
                  f'z[{z_edges[i]:.2f},{z_edges[i+1]:.2f})  '
                  f'ng[{ng_edges[j]:.0f},'
                  f'{ng_edges[j+1] if np.isfinite(ng_edges[j+1]) else float("inf"):.0f})  '
                  f'Δm[{stacks[name]["dm_lo"]:+.2f},{stacks[name]["dm_hi"]:+.2f}]  '
                  f'N={len(bkgs):,}  B_outer={boosts[-1]:.3f}±{b_err_outer:.3f}  '
                  f'({time.time()-t0:.1f}s)')

b_outer = np.array([v for _, v, _ in boost_outer])
print(f'\nOuter-bin boost factor over {len(boost_outer)} stacks:')
print(f'  median = {np.median(b_outer):.3f}  mean = {b_outer.mean():.3f}  '
      f'min = {b_outer.min():.3f}  max = {b_outer.max():.3f}')
if not (0.85 < np.median(b_outer) < 1.15):
    print('  WARNING: outer-bin boost median far from 1 — check binning / random count')

print(f'\nProduced {len(stacks)} / 27 stacks')

### 7.1 — Boost-factor diagnostic

$B(R)$ for all 27 stacks. The boost has already been applied to `sigmas` and `sigmas_cov` above; here we inspect it directly. Boost should be $>1$ at small $R$ (cluster-member dilution) and approach 1 at the outer radii. Anomalous outer-bin values flag a possible mismatch between source coverage and the random catalog used internally by `stacked_signal`.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 12), sharex=True, sharey=True)
for i in range(3):
    for j in range(3):
        ax = axes[i, j]
        for k in range(3):
            name = f'z{i}_ng{j}_dm{k}'
            if name not in stacks: continue
            s = stacks[name]
            b_err = np.sqrt(np.diag(s['boosts_cov']))
            ax.errorbar(radii, s['boosts'], yerr=b_err,
                        fmt='o-', ms=4, capsize=2.5, color=colors[k],
                        label=(f'Δm[{s["dm_lo"]:+.2f},{s["dm_hi"]:+.2f}]  '
                               f'N={s["n_stacked"]:,}'))
        ax.axhline(1.0, color='k', lw=1.0, ls='--', alpha=0.6)
        ax.set_xscale('log')
        if i == 2:
            ax.set_xlabel(r'$R$ [Mpc]')
        if j == 0:
            ax.set_ylabel(r'Boost $B(R)$')
        hi = (f'{ng_edges[j+1]:.0f}'
              if np.isfinite(ng_edges[j+1]) else r'\infty')
        ax.set_title(
            rf'$z\in[{z_edges[i]:.2f},{z_edges[i+1]:.2f})$, '
            rf'$N_{{\rm gals}}\in[{ng_edges[j]:.0f},{hi})$', fontsize=10)
        ax.grid(alpha=0.3, which='both')
        ax.legend(fontsize=7, loc='upper right')
plt.suptitle('Boost factors per stack with bootstrap errors (should → 1 at outer $R$)',
             fontsize=12, y=1.005)
plt.tight_layout()
plt.show()

## 8 — Save & sample plots

In [9]:
with open(RESULTS_PKL, 'wb') as fh:
    pickle.dump({
        'radii':             radii,
        'bins_lims':         BINS_LIMS,
        'nboot':             NBOOT,
        'radius_mpc':        RADIUS_MPC,
        'z_edges':           z_edges,
        'ng_edges':          ng_edges,
        'dm_edges_per_cell': dm_edges_per_cell,
        'stacks':            stacks,
    }, fh)
print(f'Wrote {len(stacks)} stacks to {RESULTS_PKL}')

Wrote 27 stacks to data/wazp_stacked_signals.pkl


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 12),
                         sharex=True, sharey=True)
colors = ['steelblue', 'darkorange', 'crimson']  # dm0 / dm1 / dm2

for i in range(3):
    for j in range(3):
        ax = axes[i, j]
        for k in range(3):
            name = f'z{i}_ng{j}_dm{k}'
            if name not in stacks:
                continue
            s   = stacks[name]
            err = np.sqrt(np.diag(s['sigmas_cov']))
            ax.errorbar(
                radii, s['sigmas'], yerr=err,
                fmt='o-', ms=4, capsize=2.5, color=colors[k],
                label=(f'Δm[{s["dm_lo"]:+.2f},{s["dm_hi"]:+.2f}]  '
                       f'N={s["n_stacked"]:,}'),
            )
        ax.set_xscale('log')
        ax.set_yscale('log')
        if i == 2:
            ax.set_xlabel(r'$R$  [Mpc]')
        if j == 0:
            ax.set_ylabel(r'$\Delta\Sigma$  [$M_\odot/\mathrm{pc}^2$]')
        hi = (f'{ng_edges[j+1]:.0f}'
              if np.isfinite(ng_edges[j+1]) else r'\infty')
        ax.set_title(
            rf'$z\in[{z_edges[i]:.2f},{z_edges[i+1]:.2f})$, '
            rf'$N_{{\rm gals}}\in[{ng_edges[j]:.0f},{hi})$',
            fontsize=10,
        )
        ax.legend(fontsize=7, loc='upper right')
        ax.grid(alpha=0.3, which='both')

plt.suptitle('WaZP stacked ΔΣ — 27 stacks (3 z × 3 NGALS × 3 per-cell Δm tertiles)',
             fontsize=12, y=1.005)
plt.tight_layout()
plt.show()